# Module 6.5 — Cross-Dataset Deployment to Taiwan

**Module 6: Vision + Deployment | Week 2 | Lecture (Session 6.5)**

*Source: New content (Session 31, 2026-05-18). Empirical core from [`M05-CROSS-DATASET-INVESTIGATION-FINDINGS.md`](../../M05-CROSS-DATASET-INVESTIGATION-FINDINGS.md) (Session 16, 2026-05-12) + [`M05-AAC-INVESTIGATION-FINDINGS.md`](../../M05-AAC-INVESTIGATION-FINDINGS.md). Lecture is tabular-only — same 6-feature SHARED trunk as M05-lab4. Photo-augmented multi-task on Taiwan is a natural extension but not empirically validated; tabular Spearman 0.927 is the load-bearing finding.*

---

This is the lecture M05 has been building toward. We trained a shared-trunk model on PetFinder (Malaysia) and AAC (Texas). Today we point it at Taiwan — a country it's never seen — and ask: does the model know which shelters have the hardest-to-place dogs?

The answer: yes, almost perfectly. When we rank Taiwan's shelters by the model's predicted adoption rate and compare that ranking to the real published government rates, the correlation is 0.927 out of 1.0. The model gets the order nearly right despite never seeing Taiwan data.

By the end of this lecture you should be able to say, with evidence:

- The **multi-task PF head** gets Taiwan's shelter ranking nearly perfect (Spearman 0.927) — better than any single-dataset model we tested.
- The model has never seen Taiwan data. Multi-task training forced it to learn what's *invariant* between PetFinder and AAC; that invariant structure also describes Taiwan.
- The catch: the model tells you which **shelters** need help, not which **individual dogs** will be adopted. That distinction is the lesson.

## Glossary

- **Spearman ρ (rho)** — rank correlation. "If I order the shelters by predicted adoption rate, does that match the order by published rate?" 1.0 = perfect agreement; 0.0 = random. This is how we know the model gets the *order* right even when the absolute numbers are off.
- **Ranking-only validity** — the model's *ordering* (high to low) is useful even when its absolute predictions are biased. Good for triage ("which shelters need support first?"); not good for thresholded decisions ("should we transfer this dog?").
- **Selection bias on COA** — Taiwan COA's open-data feed lists *current stock*. Fast-adopting animals leave the listings quickly, so the snapshot skews toward harder-to-adopt animals. The model's under-prediction is partially correct — the easy cases already left.

## The same architecture, three ways

Same architecture you built in M05-lab4, same architecture from M06-3. The only thing that changes today is *what we do with it* — we deploy it to Taiwan.

| Module | Application | Trunk input | Heads | What the heads do |
|---|---|---|---|---|
| **M05-lab4** | Cross-dataset (PF ↔ AAC) | 6 SHARED tabular features | PF head, AAC head | Each consumes trunk_out + dataset-specific features; trains on dataset-specific labels |
| **M06-3 / 6.4** | Single-dataset aux supervision (PF) | 768 multi-photo CNN features | Aux head, adopt head | Aux predicts is_young; adopt consumes trunk_out + tab_6 |
| **M06-5 (today)** | **Cross-dataset deployment (PF + AAC → Taiwan)** | **6 SHARED tabular features** | **PF head, AAC head** | **Same architecture as M05-lab4; deployed by scoring Taiwan COA through the PF head** |

The novelty isn't a new architecture — it's the **deployment**. We trained on PF + AAC, we score on Taiwan COA. The multi-task framework's job is to learn what's invariant between PetFinder and AAC; the hope is that invariant structure also describes Taiwan.

Photos aren't in today's architecture. The Spearman 0.927 finding used tabular SHARED features only. Adding photos on the trunk is a natural extension but isn't empirically validated; the documented number is for the tabular version, so that's what we ship.

In [ ]:
import os
import sys
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr

PF_URL  = 'CONFIGURE_ME'     # ← petfinder-mini URL
AAC_URL = 'CONFIGURE_ME'     # ← aac-mini URL
PF_DIR  = '/content/petfinder'
AAC_DIR = '/content/aac'

def fetch(url, target_dir, marker_file, label):
    if os.path.exists(f'{target_dir}/{marker_file}'):
        return
    if url == 'CONFIGURE_ME':
        raise RuntimeError(f'Set {label} above to your instructor URL.')
    os.makedirs(target_dir, exist_ok=True)
    import gdown
    zip_path = f'{target_dir}/_bundle.zip'
    gdown.download(url=url, output=zip_path, quiet=False, fuzzy=True)
    if not os.path.exists(zip_path):
        raise RuntimeError(f'{label} download failed — check URL and Drive share settings.')
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(target_dir)
    os.remove(zip_path)
    if not os.path.exists(f'{target_dir}/{marker_file}'):
        raise RuntimeError(f'{marker_file} not found after extraction — wrong zip?')

fetch(PF_URL,  PF_DIR,  'train.csv',    'PF_URL')
fetch(AAC_URL, AAC_DIR, 'aac_prep.py',  'AAC_URL')

sys.path.insert(0, PF_DIR)
sys.path.insert(0, AAC_DIR)
from aac_prep import load_aac

device = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)
print(f'Bundles ready; device={device}')



In [ ]:
# --- PetFinder ---
pf = pd.read_csv(f'{PF_DIR}/train.csv')
pf['adopted_in_30_days'] = (pf['AdoptionSpeed'] <= 2).astype(int)
pf['age_adult']      = (pf['Age'] >= 12).astype(int)
pf['is_young']       = (pf['Age'] < 6).astype(int)
pf['is_mixed_breed'] = ((pf['Breed1'] == 307) | (pf['Breed2'] != 0)).astype(int)
pf['gender_male']    = (pf['Gender'] == 1).astype(int)
pf['gender_female']  = (pf['Gender'] == 2).astype(int)
pf['sterilized_yes'] = (pf['Sterilized'] == 1).astype(int)

# --- AAC ---
aac = load_aac()
# Per-species median-cutoff target (T_AAC from Session 16: dogs 7.21d, cats 25.82d).
T_AAC = {1: 7.21, 2: 25.82}
fast = np.zeros(len(aac), dtype=int)
for sid, T in T_AAC.items():
    m = (aac['Type'] == sid)
    fast[m & (aac['outcome_type'] == 'Adoption') & (aac['days_in_shelter'] <= T)] = 1
aac['adopted_fast_AAC'] = fast

# --- Taiwan COA ---
# COA ships in petfinder-mini.zip; schema is mostly Chinese-labeled categoricals.
coa_raw = pd.read_csv(f'{PF_DIR}/COA_OpenData.csv')

def encode_coa(df):
    """Map COA's Chinese-labeled columns onto the SHARED feature schema."""
    out = pd.DataFrame()
    out['Type']           = df['animal_kind'].map({'狗': 1, '貓': 2})
    out['age_adult']      = df['animal_age'].isin(['ADULT', 'SENIOR']).astype(int)
    out['is_young']       = (df['animal_age'] == 'CHILD').astype(int)
    out['is_mixed_breed'] = df['animal_Variety'].fillna('').str.contains('混種').astype(int)
    out['gender_male']    = (df['animal_sex'] == 'M').astype(int)
    out['gender_female']  = (df['animal_sex'] == 'F').astype(int)
    out['sterilized_yes'] = (df['animal_sterilization'] == 'T').astype(int)
    out['shelter_name']   = df['shelter_name']
    return out

coa = encode_coa(coa_raw).dropna(subset=['Type'])
coa['Type'] = coa['Type'].astype(int)

print(f'PetFinder: {len(pf):,} rows  (dogs={(pf["Type"]==1).sum():,}, cats={(pf["Type"]==2).sum():,})')
print(f'AAC:       {len(aac):,} rows  (dogs={(aac["Type"]==1).sum():,}, cats={(aac["Type"]==2).sum():,})')
print(f'Taiwan COA: {len(coa):,} rows (dogs={(coa["Type"]==1).sum():,}, cats={(coa["Type"]==2).sum():,})')

## Distribution shift — what we're working against

When you take a model trained on one dataset and deploy it on another, differences in the populations cause problems. Here are the biggest gaps between PetFinder (Malaysia) and AAC (Texas):

| Quantity | PF dogs | AAC dogs | PF cats | AAC cats |
|---|---|---|---|---|
| `adopted_in_30` rate | 46.3% | 34.1% | 54.9% | **20.7%** |
| `age_adult` | 29.6% | **68.7%** | 19.6% | 33.1% |
| `is_young` | 61.6% | **22.7%** | 67.7% | 62.3% |
| `is_mixed_breed` | 86.5% | 74.1% | 25.2% | **97.6%** |

AAC dogs are much older — PetFinder lists dogs when they're posted online (often young), while AAC records dogs at municipal intake (often strays, often adults). AAC cats are 97.6% "Mix" because Austin's naming convention labels almost everything as mixed breed, which destroys that feature's predictive value. AAC cat adoption rate is 34 points below PetFinder cats. **A PF model deployed on AAC predicts adoption rates calibrated for the wrong population.**

**Bottom line:** PetFinder is Malaysian online listings. AAC is a Texas municipal shelter. Taiwan is a third context entirely. The multi-task model's advantage is that it already learned to handle the PF-vs-AAC gap — maybe that generalization extends to Taiwan. That's what we test next.

In [ ]:
# Same MultiDatasetMLP you built in M05-lab4. Reusing as-given.
from m05_labbench import _make_layers

SHARED = ['age_adult', 'is_young', 'is_mixed_breed', 'gender_male', 'gender_female', 'sterilized_yes']

class MultiDatasetMLP(nn.Module):
    """Shared trunk + per-dataset heads. PF head and AAC head each take their own specific features.

    Today we use **only the SHARED features** as the trunk input and feed the
    SHARED features into both heads as well (no PF-specific or AAC-specific
    feature block). That matches the Session 16 Taiwan-deployment recipe —
    Taiwan COA can't supply PF-specific features (PhotoAmt, Fee) at scoring
    time, so training a PF head that depends on them would break deployment.
    """

    def __init__(self, n_shared, shared_hidden=(64, 32), head_hidden=(16,), dropout=0.2):
        super().__init__()
        trunk_layers, trunk_out = _make_layers(n_shared, shared_hidden, dropout)
        self.trunk = nn.Sequential(*trunk_layers)
        self.trunk_dim = trunk_out
        pf_layers, pf_h = _make_layers(trunk_out, head_hidden, dropout)
        pf_layers.append(nn.Linear(pf_h, 1))
        self.head_pf = nn.Sequential(*pf_layers)
        aac_layers, aac_h = _make_layers(trunk_out, head_hidden, dropout)
        aac_layers.append(nn.Linear(aac_h, 1))
        self.head_aac = nn.Sequential(*aac_layers)

    def forward_pf(self, x_shared):
        return self.head_pf(self.trunk(x_shared)).squeeze(-1)

    def forward_aac(self, x_shared):
        return self.head_aac(self.trunk(x_shared)).squeeze(-1)

    def trunk_only(self, x_shared):
        return self.trunk(x_shared)


In [ ]:
# Training helpers. The PF head trains on PF labels; the AAC head trains on AAC labels;
# both heads share the trunk via the combined loss.
def to_t(arr):
    return torch.tensor(arr, dtype=torch.float32, device=device)


def prep_species(species_id):
    """Extract SHARED features + targets for one species, train/test split per dataset."""
    pf_s = pf[pf['Type'] == species_id].copy()
    aac_s = aac[aac['Type'] == species_id].copy()

    X_pf = pf_s[SHARED].values.astype(np.float32)
    y_pf = pf_s['adopted_in_30_days'].values
    X_aac = aac_s[SHARED].values.astype(np.float32)
    y_aac = aac_s['adopted_fast_AAC'].values

    pf_tr, pf_te = train_test_split(np.arange(len(y_pf)),  test_size=0.25, random_state=42, stratify=y_pf)
    aac_tr, aac_te = train_test_split(np.arange(len(y_aac)), test_size=0.25, random_state=42, stratify=y_aac)

    # Single scaler fit on the union of PF and AAC train rows so both heads see one coordinate system.
    scaler = StandardScaler().fit(np.vstack([X_pf[pf_tr], X_aac[aac_tr]]))
    return {
        'X_pf_tr':   scaler.transform(X_pf[pf_tr]).astype(np.float32),
        'X_pf_te':   scaler.transform(X_pf[pf_te]).astype(np.float32),
        'y_pf_tr':   y_pf[pf_tr],  'y_pf_te':   y_pf[pf_te],
        'X_aac_tr':  scaler.transform(X_aac[aac_tr]).astype(np.float32),
        'X_aac_te':  scaler.transform(X_aac[aac_te]).astype(np.float32),
        'y_aac_tr':  y_aac[aac_tr], 'y_aac_te':  y_aac[aac_te],
        'scaler':    scaler,
    }


def train_multitask(data, seed, alpha=0.5, epochs=100):
    torch.manual_seed(seed)
    model = MultiDatasetMLP(n_shared=len(SHARED)).to(device)
    opt = optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.BCEWithLogitsLoss()
    Xpf_tr, Xaac_tr = to_t(data['X_pf_tr']), to_t(data['X_aac_tr'])
    ypf_tr, yaac_tr = to_t(data['y_pf_tr']), to_t(data['y_aac_tr'])
    for _ in range(epochs):
        model.train(); opt.zero_grad()
        loss = alpha * loss_fn(model.forward_pf(Xpf_tr),  ypf_tr) + \
               alpha * loss_fn(model.forward_aac(Xaac_tr), yaac_tr)
        loss.backward()
        opt.step()
    return model


def train_pf_only(data, seed, epochs=100):
    """Single-headed MLP on PF SHARED features — the §12 baseline."""
    torch.manual_seed(seed)
    from torch.nn import Sequential, Linear, ReLU, Dropout
    n = len(SHARED)
    layers, prev = [], n
    for h in (64, 32):
        layers += [Linear(prev, h), ReLU(), Dropout(0.2)]; prev = h
    layers.append(Linear(prev, 1))
    model = Sequential(*layers).to(device)
    opt = optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.BCEWithLogitsLoss()
    Xpf_tr, ypf_tr = to_t(data['X_pf_tr']), to_t(data['y_pf_tr'])
    for _ in range(epochs):
        model.train(); opt.zero_grad()
        loss_fn(model(Xpf_tr).squeeze(-1), ypf_tr).backward()
        opt.step()
    return model

In [ ]:
# Train multi-task + PF-only on dogs (3 seeds). Cats deferred to the 6.6 lab.
SEEDS = [42, 7, 2026]
data_dogs = prep_species(1)

mt_models, pf_models = [], []
for s in SEEDS:
    mt_models.append(train_multitask(data_dogs, s))
    pf_models.append(train_pf_only(data_dogs, s))

# Quick sanity check: PF→AAC cross-dataset AUC via the multi-task PF head (zero-pad).
def pf_head_auc_on_aac(models, data):
    Xaac_te = to_t(data['X_aac_te'])
    aucs = []
    for m in models:
        m.eval()
        with torch.no_grad():
            probs = torch.sigmoid(m.forward_pf(Xaac_te)).cpu().numpy()
        aucs.append(roc_auc_score(data['y_aac_te'], probs))
    return float(np.mean(aucs))

def pf_only_auc_on_aac(models, data):
    Xaac_te = to_t(data['X_aac_te'])
    aucs = []
    for m in models:
        m.eval()
        with torch.no_grad():
            probs = torch.sigmoid(m(Xaac_te).squeeze(-1)).cpu().numpy()
        aucs.append(roc_auc_score(data['y_aac_te'], probs))
    return float(np.mean(aucs))

print(f'Multi-task PF head → AAC dogs  AUC: {pf_head_auc_on_aac(mt_models, data_dogs):.3f}  (Session 16: 0.610)')
print(f'PF-only            → AAC dogs  AUC: {pf_only_auc_on_aac(pf_models, data_dogs):.3f}  (Session 16: 0.595)')

# Trunk-only linear probe — the real cross-dataset signal.
Xaac_tr_t = to_t(data_dogs['X_aac_tr']); Xaac_te_t = to_t(data_dogs['X_aac_te'])
trunk_aucs = []
for m in mt_models:
    m.eval()
    with torch.no_grad():
        z_tr = m.trunk_only(Xaac_tr_t).cpu().numpy()
        z_te = m.trunk_only(Xaac_te_t).cpu().numpy()
    lr = LogisticRegression(max_iter=1000).fit(z_tr, data_dogs['y_aac_tr'])
    trunk_aucs.append(roc_auc_score(data_dogs['y_aac_te'], lr.predict_proba(z_te)[:, 1]))
print(f'Multi-task trunk + LR on AAC dogs labels  AUC: {float(np.mean(trunk_aucs)):.3f}  (Session 16: 0.661, +0.066 over §12)')

The trunk's hidden representation — the 32 numbers it learns to encode about each dog — transfers across datasets. A simple logistic regression trained on those 32 numbers, using AAC's labels, beats the PetFinder-only model's cross-dataset AUC by +0.066. That's the shared trunk doing its job: it learned features useful for predicting adoption in *both* PetFinder and AAC, not just one.

The PF head's cross-dataset AUC (~0.610) also beats PF-only (~0.595), but the trunk-LR result is the cleaner signal — it isolates the trunk's contribution from the head's.

In [ ]:
# Score Taiwan COA via both models. COA gives us no per-animal labels — only
# aggregates (mean predicted P, IQR) and per-shelter Spearman vs published rates.
coa_dogs = coa[coa['Type'] == 1].copy()
X_coa = data_dogs['scaler'].transform(coa_dogs[SHARED].values).astype(np.float32)
X_coa_t = to_t(X_coa)

def score_coa(models, head):
    """head: 'pf', 'aac', or 'plain' for the single-headed PF-only baseline."""
    probs_per_seed = []
    for m in models:
        m.eval()
        with torch.no_grad():
            if head == 'pf':
                logits = m.forward_pf(X_coa_t)
            elif head == 'aac':
                logits = m.forward_aac(X_coa_t)
            else:  # 'plain' = PF-only single-headed baseline
                logits = m(X_coa_t).squeeze(-1)
            probs_per_seed.append(torch.sigmoid(logits).cpu().numpy())
    return np.mean(probs_per_seed, axis=0)

p_mt_pf  = score_coa(mt_models, 'pf')
p_mt_aac = score_coa(mt_models, 'aac')
p_pf     = score_coa(pf_models, 'plain')

def aggregate(probs, label):
    return {
        'model':     label,
        'mean':      float(np.mean(probs)),
        'gap_target': float(np.mean(probs) - 0.453),
        'median':    float(np.median(probs)),
        'IQR':       float(np.quantile(probs, 0.75) - np.quantile(probs, 0.25)),
        '%>0.5':     float((probs > 0.5).mean()),
    }

df_agg = pd.DataFrame([
    aggregate(p_pf,     'PF-only'),
    aggregate(p_mt_aac, 'Multi-task AAC head'),
    aggregate(p_mt_pf,  'Multi-task PF head'),
])
print(df_agg.to_string(index=False, float_format='%.4f'))

### What the numbers tell you

The table above answers one question well and one question badly.

**The question it answers well:** "Which Taiwan shelters have the hardest-to-place populations?" The multi-task model's predictions vary enough across shelters to make real distinctions — its IQR (~0.213) is about twice PF-only's (~0.109).

**The question it answers badly:** "Will *this specific dog* get adopted in 30 days?" The model under-predicts overall (mean ~0.365 vs Taiwan's actual ~0.453), and we have no per-animal Taiwan labels to check individual accuracy. Understanding which question you're answering — and which you're not — is the whole point of this lecture.

Three patterns in the numbers (your results should land within ±0.02):

- **Multi-task PF head's mean (~0.365) is closest to the real Taiwan target (~0.453).** PF-only is close behind (~0.357). The model that trained on two shelters' data gives the most realistic Taiwan average.
- **AAC-only collapses.** Mean ~0.172, essentially predicting every dog is hard to adopt. Why? AAC's predictive signal lives in features like intake type and intake condition — features Taiwan COA doesn't have. With only the 6 SHARED features, the AAC head has nothing to work with.
- **Multi-task PF head has ~2× the IQR of PF-only.** The multi-task model makes more distinct calls — wider use of the [0, 1] interval. Better for ranking animals even when the absolute probabilities are off.

In [ ]:
# Per-shelter Spearman vs published Taiwan city rates.
# Top 7 shelters by listing count (from M05-CROSS-DATASET-INVESTIGATION-FINDINGS §5).
TOP_SHELTERS = {
    '新北市八里區公立動物之家': 0.95,   # New Taipei, urban
    '新北市五股區公立動物之家': 0.95,
    '新北市新店區公立動物之家': 0.95,
    '臺北市動物之家':           0.88,   # Taipei City
    '高雄市燕巢動物保護關愛園區': 0.77,  # Kaohsiung
    '南投縣公立動物收容所':     0.47,   # Nantou, rural
    '雲林縣流浪動物收容所':     0.39,   # Yunlin, rural
}

rows = []
for shelter, published_rate in TOP_SHELTERS.items():
    m = coa_dogs['shelter_name'] == shelter
    if m.sum() == 0:
        continue
    rows.append({
        'Shelter':         shelter,
        'n_listings':      int(m.sum()),
        'Published rate':  published_rate,
        'PF-only pred':    float(p_pf[m].mean()),
        'Multi-task pred': float(p_mt_pf[m].mean()),
    })
shelter_df = pd.DataFrame(rows)
print(shelter_df.to_string(index=False, float_format='%.3f'))
print()
spear_pf = spearmanr(shelter_df['Published rate'], shelter_df['PF-only pred']).statistic
spear_mt = spearmanr(shelter_df['Published rate'], shelter_df['Multi-task pred']).statistic
print(f'Spearman vs published city rates:')
print(f'  PF-only:    {spear_pf:.3f}  (Session 16: 0.778)')
print(f'  Multi-task: {spear_mt:.3f}  (Session 16: 0.927)')

### Why Spearman matters here

The model has never seen Taiwan data and has no per-animal Taiwan labels to validate against. So we can't ask "is the model right?" in the usual accuracy / AUC sense. What we *can* ask is: does the model's *ordering* of Taiwan shelters match the real ordering?

PF-only's Spearman of ~0.78 is good — it picks up urban-vs-rural directionally. Multi-task's ~0.93 is *very* good — it gets nearly the right per-shelter ranking despite never seeing Taiwan data.

**What Spearman 0.93 means operationally.** A shelter-network coordinator looking at the model's per-shelter predictions would prioritize support correctly almost every time. "Which shelters need help with adoption logistics?" gets the right answer.

**What it doesn't mean.** Spearman tells us the model ranks *shelters* correctly, not *individual dogs*. The published Taiwan rates are aggregate — they tell us nothing about whether the model correctly orders the 967 animals inside a single shelter. Per-animal validation would require per-animal Taiwan outcome labels, which COA doesn't publish.

## Deployment lessons

The standard cross-dataset rule — "don't deploy without retraining" — is too simple. This lecture shows that *what kind of decision you're making* determines whether cross-dataset deployment works:

| Use case | Cross-dataset viability | Why |
|---|---|---|
| Per-animal yes/no decisions | **Don't deploy** | Ranking too weak to beat class imbalance |
| Ranking / triage / featuring order | **Modest utility** (AUC ~0.6 on AAC) | Within-species ranking partially preserved |
| Aggregate per-shelter planning | **Strong utility** (Spearman 0.93) | Multi-task preserves geographic structure |
| Per-animal absolute probability | **Don't deploy** | Under-predicts by ~9 points; selection bias adds more uncertainty |

Three findings that got us here:

- **Multi-task wins.** Best aggregate calibration (gap −0.088), best spread (IQR 0.213), best shelter ranking (Spearman 0.927). The shared trunk learned features useful for *both* PetFinder and AAC — that representation generalizes to Taiwan more cleanly than either single-dataset model.
- **AAC-only collapses on Taiwan.** AAC's predictive signal lives in intake-type and intake-condition features — features Taiwan COA doesn't have. With only the 6 SHARED features, the AAC head is effectively random. Same goal (predict adoption), fundamentally different signal.
- **Fine-tuning on AAC didn't help either.** We tested training on PetFinder first, then fine-tuning on a small AAC sample. The fine-tuned model barely changed (+0.004 dogs / −0.002 cats over the original). That told us to stop trying fancier transfer methods — multi-task training, not sequential fine-tuning, is the answer.

**The under-prediction caveat.** The model predicts a mean of ~0.357 for Taiwan, but the published target is ~0.453 — a gap of about 9 points. Part of this is real: Taiwan COA's listing snapshot shows dogs *currently waiting*, not fresh listings. Dogs that get adopted quickly leave the snapshot fast, so what remains skews toward harder-to-adopt animals. The model was trained on PetFinder's fresh listings, so it correctly reads the remaining COA population as harder cases. The under-prediction isn't entirely a bug — but it does mean you shouldn't use the raw probabilities as adoption forecasts.

---

## Summary

- **Multi-task PF head is the Taiwan deployment recipe.** Spearman 0.927 vs published city rates; the model has never seen Taiwan data and still gets the shelter ranking nearly right.
- **The deployment answers "which shelters need help," not "will this dog get adopted."** Shelter-level planning and rank-based triage work; per-animal yes/no decisions don't.
- **The model under-predicts, and that's partly correct.** Taiwan COA's snapshot only shows dogs still waiting — the easy cases already left. The model reads them as harder cases because they are.

What happens when we add Taiwan *photos* to this model? 6.7 answers — and the finding is the strongest teaching moment in M06.

### Where M06 goes from here

- **6.6 — Lab:** You reproduce the multi-task PF head Taiwan recipe end-to-end on bundled data. Validate Spearman 0.927 on your hardware. Build the per-shelter table. Stretch: replicate the four-model comparison (including AAC-only and pretrain-finetune) to see the failure modes yourself.
- **6.7 — Photo quality + responsible vision-ML:** What the CNN actually encodes, when photos help vs hurt, the photo-quality gap as an actionable signal, and the M07 handoff.
- **6.8 — Pre-capstone lab:** Build your Taiwan dog's photo-augmented profile.
- **6.9 — Capstone:** The Taiwan adoption brief itself, 5 sections — the deliverable M05 was promising and M06 makes good on.

---

**Next session (Lab 6.6):** Reproduce the Taiwan deployment recipe on bundled data. Same architecture, your hands on it, the headline number reproduced on your machine.